# **Week 2: Natrual Language Processing and Word Embeddings**

Natural language processing with deep learning is a powerful combination. Using word vector representations and embedding layers, train recurrent neural networks with outstanding performance across a wide variety of applications, including sentiment analysis, named entity recognition and neural machine translation.

**Learning Objectives:**

- Explain how word embeddings capture relationships between words
- Load pre-trained word vectors
- Measure similarity between word vectors using cosine similarity
- Use word embeddings to solve word analogy problems such as Man is to Woman as King is to ______.
- Reduce bias in word embeddings
- Create an embedding layer in Keras with pre-trained word vectors
- Describe how negative sampling learns word vectors more efficiently than other methods
- Explain the advantages and disadvantages of the GloVe algorithm
- Build a sentiment classifier using word embeddings
- Build and train a more sophisticated classifier using an LSTM

---

## **Table of Contents**

---

## **Word Representation**

This section introduces a fundamental shift in how Deep Learning models process language, moving from simple, isolated word counts to rich, "featurized" representations. The core message is that **Word Embeddings** allow algorithms to generalize knowledge by understanding that certain words (like "apple" and "orange") are semantically similar.

### **High-level Summary**

In early NLP, computers treated words as discrete symbols, much like ID numbers in a database. This approach, known as **One-Hot Encoding**, failed because it lacked a mathematical "bridge" between related concepts. Word Embeddings solve this by placing words in a continuous, high-dimensional space where "meaning" is defined by coordinates. If "apple" and "orange" are close together in this space, the model can finally understand that what it learns about one likely applies to the other.

### **Key Takeaways**

* **The Limitations of One-Hot Vectors:**
    * Previously, words were represented as vectors with a single "1" in a specific position (e.g., $[0, 0, 1, 0]$).
    * The inner product between any two different one-hot vectors is always **0**. Consequently, the model mathematically "sees" no more similarity between "apple" and "orange" than it does between "apple" and "king." To the algorithm, every word is a total stranger to every other word.

* **Featurized Representations:**
    * By assigning a dense vector (e.g., 300 dimensions) to each word, we can represent shared characteristics like "Gender", "Royalty", "Age", or "Food status".
    * For example, "Man" might have a gender feature value of **-1**, while "Woman" has **+1**. "Apple" and "Orange" would both have high values for the "Food" feature but neutral values for "Royalty."

    <div align='center'>
        <img src='assets/word_embeddings.png' width=750px>
    </div>
    <br>
    
* **Generalization & Analogies:**
    * Embeddings enable models to solve analogies via vector arithmetic (e.g., $man:woman :: king:queen$).
    * If a model learns that "orange juice" is a common phrase, the shared features in the embedding space allow it to automatically infer that "apple juice" is also a valid and likely phrase, even if it has never seen that specific combination before.

* **Visualizing High Dimensions:**
    * Since we cannot visualize 300D space, algorithms like **t-SNE** or **PCA** are used to compress these embeddings into 2D plots.
    * In these visualizations, related concepts (animals, fruits, numbers) naturally cluster together, providing visual proof that the model has "learned" semantic relationships.
    
    <div align='center'>
        <img src='assets/word_embeddings_visualization.png' width=500px>
    </div>

* **Addressing Bias:**
    * Because these models learn from real-world text, they can inherit human biases related to gender, ethnicity, or socioeconomic status.
    * To resolve this problem, modern NLP involves **debiasing** (neutralizing and equalizing) these vectors to ensure the model doesn't reinforce harmful stereotypes, such as associating specific professions with only one gender.

### **One-Hot vs. Embeddings: A Comparison**

| Feature | One-Hot Encoding | Word Embeddings |
| :--- | :--- | :--- |
| **Sparsity** | Sparse (mostly zeros) | Dense (filled with floats) |
| **Vector Size** | Vocabulary size (e.g., 50,000+) | Fixed size (e.g., 300) |
| **Similarity** | **0** for all word pairs | Higher for related words |
| **Relationship** | Independent symbols | Semantic "neighbors" |
| **Generalization** | Impossible | Built-in |

---

## **Using Word Embeddings**

This section highlights how **Word Embeddings** act as a catalyst for **Transfer Learning** in NLP. The core idea is that by learning word relationships from massive, unlabeled datasets, we can significantly improve performance on specific tasks where labeled data is scarce.

### **High-level Summary**

The primary utility of word embeddings is their ability to bridge the gap between a model's lack of world knowledge and the specific requirements of an NLP task. 

Let's consider the following two sentences:

1. "Sally Robinson is an orange farmer".
2. "Robert Lin is a durian cultivator".

In a typical scenario, a model might struggle to identify "Robert Lin" as a person's name if it follows the phrase "durian cultivator"—simply because those words are rare. However, because word embeddings are "pre-trained" on nearly the entire internet, the model has already learned that a durian is a fruit (like an orange) and a cultivator is a profession (like a farmer). By swapping **one-hot vectors** for these **dense embeddings**, we "transfer" that massive internet-scale knowledge into our small, local project.

This process is fundamentally a form of transfer learning: we leverage a "Task A" (learning word relationships from billions of unlabeled sentences) to solve a "Task B" (labeling names in a few thousand sentences). While similar to **Face Encodings** in vision, the "vocabulary" approach in NLP makes word embeddings uniquely efficient for processing text with a fixed dictionary.


### **Key Takeaways**

* **Generalization through Semantic Similarity:**
    * Embeddings allow models to recognize that "durian cultivator" is conceptually similar to "orange farmer," even if the specific words "durian" or "cultivator" were never seen during the task's training phase.

* **The Three-Step Transfer Learning Process:**
    1.  **Pre-training:** Learn embeddings from a massive unlabeled corpus (1B to 100B words) or download pre-trained vectors.
    2.  **Transfer:** Apply these dense, low-dimensional vectors (e.g., 300D) to a target task with a small labeled dataset.
    3.  **Fine-tuning (Optional):** Adjust the embeddings during target task training only if the labeled dataset is sufficiently large.

* **Dimensionality Advantage:**
    * Replaces sparse, 10,000-dimensional **one-hot vectors** with dense, 300-dimensional **embedding vectors**, making the input representation much more efficient and meaningful.

* **Task Suitability:**
    * Most beneficial for tasks with limited data like **Named Entity Recognition (NER)**, text summarization, and parsing.
    * Less critical for data-rich tasks like Machine Translation or Language Modeling.

* **Embeddings vs. Face Encodings:**
    * **Similarity:** Both represent complex data as fixed-length continuous vectors (encodings).
    * **Difference:** Face recognition requires a system that can encode *any new face*, whereas word embeddings typically map a *fixed vocabulary* of words to learned vectors.

---

## **Properties of Word Embeddings**

This section explores the fascinating ability of **Word Embeddings** to perform **Analogy Reasoning** using simple vector arithmetic. By representing words in a high-dimensional space (e.g., 300D), the model captures semantic relationships as geometric distances and directions.

### **High-level Summary**

The beauty of word embeddings lies in their mathematical "common sense." By analyzing billions of words, these models realize that the leap from "man" to "woman" is spatially almost identical to the leap from "king" to "queen." It isn't just memorizing definitions; it is mapping the **structure of language** onto a coordinate system ("parallelogram" relationship).

When we ask the model an analogy, we are essentially asking it to complete a parallelogram in a 300-dimensional space. To find the missing corner, we use **Cosine Similarity**, which focuses on the direction the vectors are pointing rather than just their raw length. This allows the model to correctly identify everything from geographical capitals to grammatical patterns, providing a powerful intuition that these "dense vectors" truly capture the features of the objects they represent.

### **Key Takeaways**

* **Vector Arithmetic for Analogies:**
    * The core discovery (by [Mikolov et al.](https://scottyih.org/files/rvecs.pdf)) is that the difference between word vectors often represents a specific semantic concept.
    * For example, $e_{man} - e_{woman} \approx e_{king} - e_{queen}$. In both cases, the resulting vector represents the concept of "gender."

* **The Analogy Algorithm:**
    * To solve "Man is to Woman as King is to ??", the algorithm searches for a word $w$ that satisfies the equation:
        $$e_{man} - e_{woman} \approx e_{king} - e_w$$
    * Mathematically, this is solved by finding a word $w$ that maximizes the similarity to the vector: $e_{king} - e_{man} + e_{woman}$.

<div align='center'>
    <img src='assets/analogies_with_word_embed.png' width=750px>
</div>

* **Cosine Similarity:**
    * The most common function used to measure how "close" two word vectors are is **Cosine Similarity**:

        $$\text{Similarity}(u, v) = \frac{u^T v}{||u||_2 ||v||_2}$$
    
    * This measures the cosine of the angle $\phi$ between vectors. If the angle is $0^\circ$, the similarity is $1$; if they are orthogonal ($90^\circ$), it is $0$.

    <div align='center'>
        <img src='assets/cosine_sim.png' width=750px>
    </div>
    <br>

    * Another similarity measure is negative value of square differences $-||\vec{u}  -\vec{v}||^2$. This is a less-common approach since it does not normalize for embeddings norm.

* **Broad Generalization:**
    * Embeddings can learn diverse relationships beyond gender. For example:
        * **Verb Tense:** $Go : Went :: Find : Found$
        * **Capitals:** $Ottawa : Canada :: Nairobi : Kenya$
        * **Currency:** $Yen : Japan :: Ruble : Russia$

* **Visualization Warning:**
    * While we use **t-SNE** to visualize embeddings in 2D, t-SNE is a **non-linear** mapping. We should not expect the "parallelogram" relationship of analogies to hold true in a t-SNE plot; it only reliably exists in the original high-dimensional space.

---

## **Embedding Matrix**

In this section, the focus shifts from the *why* of word embeddings to the *how*—specifically, the data structures used to store and retrieve them. The core concept introduced is the **Embedding Matrix ($E$)**, which acts as a lookup table for every word in a vocabulary.

### **High-level Summary**

Think of the **Embedding Matrix** as a giant library where every column is a book containing the "personality" of a single word. If our vocabulary has 10,000 words, we have 10,000 columns. 

When a model needs to "look up" a word like "Orange," it uses a **One-Hot Vector** as a pointer. Mathematically, when we multiply the Embedding Matrix $E$ by this pointer, the zeros in the one-hot vector cancel out every single word in the dictionary except for the one we want. The result is a clean, 300-dimensional vector that represents "Orange" in a way the neural network can actually understand.

While we represent this as matrix multiplication in theory, in practice, it’s much more efficient. Software frameworks like TensorFlow or PyTorch use specialized **"Embedding Layers"** that perform a direct index lookup (essentially just grabbing the column) to save on computation, since multiplying by a vector of zeros is technically a waste of energy!

### **Key Takeaways**

* **The Embedding Matrix ($E$):**
    * This is a massive matrix of shape $(d, V)$, where $d$ is the embedding dimension or features (e.g., **300**) and $V$ is the vocabulary size (e.g., **10,000**).
    * Each column in this matrix represents the "featurized" 300-dimensional vector for a specific word.

* **The One-Hot Vector ($O$):**
    * To find a specific word, we use a one-hot vector $O_j$, which is a $V \times 1$ vector containing a **1** at index $j$ and **0**s elsewhere.

* **Retrieval via Matrix Multiplication:**
    * Mathematically, selecting an embedding is represented as the product: $e_j = E \cdot O_j$.
    * Since the one-hot vector has only a single "1," this operation effectively "filters out" all other columns and returns only the 300-dimensional column corresponding to that specific word.

* **Notation:**
    * The resulting 300-dimensional vector is denoted as $e_j$ (or $e_{orange}$ in the example), representing the dense embedding for the $j$-th word in the vocabulary.

---

## **Learning Word Embeddings**

This section introduces how algorithms create word embeddings. It explains that while early methods were complex neural language models, researchers discovered that much simpler tasks—like predicting a word from its neighbor—can produce equally powerful word embeddings.

### **High-level Summary**

The history of word embeddings is a journey toward simplicity. Initially, researchers built full-scale neural networks to predict the next word in a sentence. To do this, the network had to "squish" words into, let's say, a 300-dimensional space. To get the answer right consistently, the network realized it was easier if it treated similar concepts (like different fruits) as being in the same "neighborhood" of that 300D space. This byproduct—the neighborhood map—is the **Embedding Matrix**.

As the field evolved, researchers realized they didn't need a complex window of four or six words to learn these relationships. They discovered that we can learn an excellent "map" of language by simply picking one random word (the context) and trying to guess another word that appears nearby (the target). This led to the creation of the **Skip-Gram** and **Word2Vec** models (see next section), which are far more computationally efficient and are the industry standards we use today.

### **Key Takeaways**

* **Learning via Language Modeling:**
    * The earliest successful embedding algorithms (like the one by [Bengio et al., (2003)](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)) were designed to solve a **Language Modeling** task: predicting the next word in a sequence (e.g., "I want a glass of orange [?]").

* **The Neural Architecture:**
    * Below is a high-level description of the language modeling approach:
        * **Input:** A fixed "historical window" of words (e.g., the previous 4 words).
        * **Embedding Layer:** Each word is converted from a one-hot vector to a 300D dense vector using a shared embedding matrix $E$.
        * **Hidden Layer:** The 4 embedding vectors are concatenated (e.g., into a 1,200D vector) and passed through a standard neural network layer.
        * **Output:** A **Softmax** layer that predicts the probability of all 10,000 words in the vocabulary being the "target" word.

    <div align='center'>
        <img src='assets/neural_lm.png' width=750px>
    </div>

* **The Incentive for Similarity:**
    * To minimize loss, the model is incentivized to give "apple" and "orange" similar embeddings. This is because both words often appear before "juice," and having similar vectors allows the model to use the same internal weights to predict the correct next word.

* **Alternative Contexts for Learning:**
    * If the goal is specifically to learn embeddings (rather than a perfect language model), researchers found they could vary the **"Context"** used to predict the **"Target"**:
        * **Words on Left and Right:** Predicting the middle word from surrounding words.
        * **Last One Word:** Predicting the next word given only the immediate predecessor.
        * **Nearby One Word (Skip-Gram):** Predicting a word that appears anywhere nearby, even if it's not the immediate next word.

---

## **Skip-gram Model**

This section introduces the **Skip-gram** model, a cornerstone of the **Word2Vec** family. It explains how a seemingly "impossible" task—predicting a random neighbor word—is actually a brilliant way to force a neural network to learn high-quality word embeddings efficiently.

### **High-level Summary**

The Skip-gram model turns the challenge of "predicting the unpredictable" into a feature. Because it is very hard to guess exactly which word appears near "orange," the model is forced to learn that "orange" shares a similar environment with "apple" or "juice." To minimize its loss, the model must map these related words to similar positions in the 300-dimensional embedding space.

While the math is elegant, the sheer size of modern vocabularies makes the standard **Softmax** a major speed bottleneck. To keep the training "comfortably efficient," researchers developed the **Hierarchical Softmax**, which aims to find the right word in a tree rather than checking a list of a million words one by one. This allows Word2Vec to scale to the massive internet-sized corpora required to learn deep semantic relationships.

### **Key Takeaways**

* **The Skip-gram Concept:**
    * Unlike language models that predict the *next* word, Skip-gram picks a **Context word (c)** and tries to predict a **Target word (t)** randomly chosen from a surrounding window (e.g., $\pm 5$ or $\pm 10$ words).
    * For example, in sentence *"I want a glass of orange juice to go along with my cereal."*, if "orange" is the context, the model might try to predict "juice", "glass", or even "my".

* **The Model Architecture:**
    1.  **Input:** A one-hot vector $O_c$ for the context word.
    2.  **Embedding:** Multiply $O_c$ by the embedding matrix $E$ to get the dense vector $e_c$.
    3.  **Output:** Feed $e_c$ into a **Softmax** layer to calculate the probability of every word in the vocabulary being the target.

* **The Softmax Equation:**
    * The probability of a target word $t$ given context $c$ is:

    $$P(t|c) = \frac{e^{\theta_t^T e_c}}{\sum_{j=1}^{10,000} e^{\theta_j^T e_c}}$$

    Where $\theta_t$ is the parameter vector associated with output word $t$.

* **The Bottleneck (Computational Cost):**
    * The main disadvantage of the basic Skip-gram is the **summation in the denominator**. Summing over a vocabulary of 10,000 to 1,000,000 words for *every* training step is computationally expensive and slow.

* **Solution: Hierarchical Softmax:**
    * Instead of a flat search, Hierarchical Softmax uses a **tree-structured classifier** (often a Huffman tree). 
    * This reduces the complexity from $O(V)$ (linear) to $O(\log V)$ (logarithmic). Common words stay near the top of the tree for faster access, while rare words like "durian" are deeper.

* **Context Sampling Strategy:**
    * To prevent the model from being dominated by extremely common words (like "the," "of," "a"), researchers use heuristics to ensure less frequent but semantically rich words (like "orange" or "durian") are sampled as contexts often enough.

---

## **Negative Sampling**

This section introduces **Negative Sampling**, the definitive solution to the computational bottleneck of the standard Word2Vec Skip-gram model. By reframing a massive multi-class problem as a simple binary "True/False" test, researchers made it possible to train high-quality embeddings on massive datasets in a fraction of the time.

### **High-level Summary**

Negative Sampling is essentially a "shortcut" that yields the same high-quality results as a full neural network but at a massive discount in processing power.

The **Softmax** approach is like looking at every single word in the vocabulary and deciding if it's related to the context. By focusing only on a tiny subset of the vocabulary at each step, the Negative Sampling model learns just as effectively. It learns that "orange" and "juice" are related not by checking every other word in the English language, but by repeatedly seeing that "orange" and "juice" (positive target) are a "Yes", while "orange" and "king" (negative target) are a "No".

### **Key Takeaways**

* **From Multi-Class to Binary classifier:**
    * Instead of asking "Which of these 10,000 words is the target?" (Softmax), the model asks: "Is this pair of words a real context-target pair?" (Logistic Regression).
    * **Positive Example:** A context word (e.g., "orange") and a word actually found in its $\pm 10$ word window ("juice"). Label = $1$.
    * **Negative Examples:** The same context word ("orange") paired with $k$ random words from the dictionary ("king," "book," "the"). Label = $0$.

    <div align='center'>
        <img src='assets/neg_sampling.png' width=400px>
    </div>

* **Computational Efficiency:**
    * In a standard 10,000-word Softmax, every training step requires updating all 10,000 sets of parameters.
    * With Negative Sampling, we only update $k+1$ binary classifiers (the 1 positive and $k$ negatives). For large datasets, $k$ is usually as small as **2 to 5** and for small datasets $k$ is usually **5 to 20**.

* **The Heuristic for Sampling ($3/4$ Power):**
    * To pick negative words, researchers found that using raw word frequency over-samples "boring" and frequent words (like "the" or "of"). Uniform sampling is also unrealistic. The solution was an emperical "middle ground formula", where words are sampled proportional to $f(w_i)^{3/4}$. This slightly increases the probability of picking less frequent words, ensuring the model stays "sharp." The model uses the Sigmoid function to predict the probability:
        
        $$P(y=1 | c, t) = \sigma(\theta_t^T e_c)$$

        $\theta_t$ is the parameter vector for the target word, and $e_c$ is the embedding for the context word.

### **Comparison of Softmax and Negative Sampling**

| Feature | Standard Softmax | Negative Sampling |
| :--- | :--- | :--- |
| **Output Type** | Probability distribution over $V$ | Binary (0 or 1) for specific pairs |
| **Updates per Step** | All $V$ parameters | Only $k+1$ parameters |
| **Best For** | Small vocabularies | Huge datasets / large vocabularies |
| **Training Speed** | $O(V)$ (Slow) | $O(k)$ (Very Fast) |

<br>

>**Bonus:** For in-depth technical details and PyTorch implementation of Negative Sampling, refer to the interactive [Jupyter Notebook](../../bonus/notebooks/skip_gram_negative_sampling_pytorch.ipynb).

---

## **GloVe Word Vectors**

This section provides a breakdown of the **GloVe (Global Vectors for Word Representation)** algorithm. While Word2Vec relies on local window sampling, GloVe focuses on the "global" statistics of how often words co-occur across an entire text corpus.

### **High-level Summary**

The fundamental shift in GloVe is moving from **sampling** (looking at individual pairs one by one) to **counting**. It builds a large co-occurrence matrix, $X$, where $X_{ij}$ represents how many times word $j$ (target) appears in the context of word $i$. The goal is to create word vectors whose dot product accurately predicts the log of these co-occurrence counts.

The beauty of GloVe lies in its simplicity. It essentially turns a complex linguistic problem into a **weighted least squares regression** problem. By looking at the "Global" state of the language (the entire $X$ matrix) rather than just "Local" snapshots (Skip-grams), it captures the macro-relationships of a language more efficiently. While we might not be able to look at a particular feature in a word embedding, the overall vector space becomes a powerful tool for sentiment analysis and analogy reasoning because the *relative* distances between words are preserved.

### **Key Takeaways**

* **The Objective Function:** The model minimizes the squared difference between the dot product of two vectors ($\theta_j^T e_i$) and the logarithm of their co-occurrence count ($\log X_{ij}$):

    $$\text{Minimize} \sum_{i,j} f(X_{ij}) (\theta_j^T e_i + b_i + b_j - \log X_{ij})^2$$

* **The Weighting Function $f(X_{ij})$:**
    * It handles the problem of $\log(0)$ by setting the weight to zero if the words never co-occur.
    * It acts as a heuristic to balance the "continuum" of word frequencies. It ensures common words (like "the," "is") don't dominate the loss, while rare words (like "durian") still receive enough weight to be meaningful.

* **Symmetry and Averaging:**
    * In GloVe, the context and target vectors ($\theta$ and $e$) are mathematically symmetric. Because they play identical roles in the equation, a common practice is to train both and then use their average ($\frac{e + \theta}{2}$) as the final word embedding.

* **The Problem of Interpretability:**
    * Even though we want embeddings to represent clear features like "Gender" or "Royalty," the algorithm cannot guarantee that any single dimension (e.g., the first row) corresponds to a human-interpretable trait. 
    * Due to potential linear transformations during optimization, a single dimension might be a "mixture" of many different semantic concepts.
    * Despite the lack of individual row interpretability, the global geometric relationships—like the **parallelogram map** used for analogies ($Man:Woman :: King:Queen$)—remain intact and highly effective.


>**Note:** To build intuition about why we take $log$ of $X_{ij}$ and use a weighting function to calculate loss, refer to the following [bonus](#bonus-the-mathematical-intuition-behind-glove-loss-function) section.

---

## **Bonus: The Mathematical Intuition Behind GloVe Loss Function**

The GloVe objective function is a carefully engineered "weighted least squares" regression. Two specific components—the **Logarithm** and the **Weighting Function**—work together to solve the massive scaling issues inherent in natural language.

### **1. Why the Logarithm?**

The use of the logarithm in the GloVe objective function is a "mathematical trick" that solves the problem of high-frequency "stop words" (like *the* or *and*) drowning out meaningful terms. Here are key benefits of applying $log$ function to co-occurence matrix:

* **Compressing the Dynamic Range:** In any large corpus, word frequencies follow **Zipf’s Law**, meaning the most frequent word occurs roughly twice as often as the second most frequent, and so on. Therefore, the co-occurrence count $X_{ij}$ for "of the" might be **1,000,000**, while "durian fruit" might be **1**. If we tried to predict raw counts, the squared error from a single mistake on "of the" would be $(10^6)^2$ times larger than a mistake on a rare word. Taking the **logarithm** squashes this astronomical range. Instead of comparing $1$ to $1,000,000$, the model compares $\log(1) = 0$ to $\log(1,000,000) \approx 13.8$. This puts rare and frequent words on a "level playing field".

* **Mapping Multiplicative Ratios to Linear Distances:** Linguistics is naturally multiplicative. We often understand word relationships through ratios of probabilities. For example:

    $$\frac{P(\text{solid} \mid \text{ice})}{P(\text{solid} \mid \text{steam})} \approx \text{Large Ratio}$$

    By using $\log(X_{ij})$, we transform these multiplicative ratios into simple vector subtractions:

    $$\log\left(\frac{A}{B}\right) = \log(A) - \log(B)$$

    This allows the dot product $\theta_j^T e_i$ to represent relative strength linearly, which is precisely why "parallelogram analogies" (like $King - Man + Woman = Queen$) work so well in the resulting vector space.

* **Theoretical Alignment with Skip-Gram:** In the Skip-gram (Word2Vec) model, the probability of a target word $j$ given context $i$ is modeled via Softmax:

    $$P(j \mid i) = \frac{\exp(\theta_j^T e_i)}{\sum \exp(\dots)}$$

    If we take the $log$ of a simplified version of this probability, we find that $\log P(j \mid i) \approx \theta_j^T e_i$. The creators of GloVe realized that if the dot product represents log-probability, and probability is proportional to the count ($X_{ij}$), then the dot product must be approximately equal to the log-count to keep the model's internal similarity scores and external counts on the same scale.


### **2. Why the Weighting Function $f(X_{ij})$?**

The weighting function is the model's final "safety valve." While the logarithm squashes the numerical range, the weighting function determines how much the model should actually care about a specific co-occurrence pair during a gradient update.

The function $f(X_{ij})$ is designed to satisfy three common-sense rules:
1.  **$f(0) = 0$:** If words never occur together, they contribute nothing (this also avoids the $\log(0)$ error).
2.  **Monotonicity:** $f(x)$ increases with $x$ so that frequent pairings carry more weight than rare ones.
3.  **The "Ceiling" Effect:** The weight plateaus for extremely frequent words so they don't dominate the entire model.

The standard heuristic is:

$$f(x) = \begin{cases} (x/x_{max})^\alpha & \text{if } x < x_{max} \\ 1 & \text{if } x \ge x_{max} \end{cases}$$

* **Boosting the rare words ($\alpha$ Power):** By setting $\alpha$ (typically $0.75$), the function gives a "boost" to mid-frequency words. The curve is steeper at the beginning, ensuring that words like "durian" get enough weight to move their vectors significantly.
* **The $x_{max}$ cap for stop words:** By setting $x_{max}$ (usually $100$), we declare that once two words have appeared together $100$ times, the relationship is "fully confirmed." Whether they appear $100$ or $1,000,000$ times, the weight stays at $1$. This prevents "stop-words" from drowning out the rest of the vocabulary.

---

## **Sentiment Classification**

In this section, we explore how word embeddings act as a **"force multiplier"** for sentiment classification. These pre-trained representations allow us to build highly accurate models even when we have a very limited amount of labeled data.

### **High-Level Summary**

Sentiment classification serves as a bridge between **unsupervised learning** (the billions of unlabeled words used to train GloVe or Word2Vec) and **supervised learning** (our specific set of star-rated reviews). 

The central advantage is "transferable knowledge." While our specific dataset might only contain 10,000 labeled reviews, our **Embedding Matrix ($E$)** has already "read" billions of words from the internet. This allows the model to understand that "excellent," "outstanding," and "superb" occupy a similar mathematical space, even if our training set only contained one of those words.

We can approach this task in two primary ways:
1.  **The "Bag-of-Words" Averaging Model:** A fast, efficient method that is unfortunately "blind" to sentence structure.
2.  **The RNN-Based Model:** A more sophisticated approach that respects the flow of language. By using an RNN, we move beyond a "mush" of vectors to a coherent model that understands how words modify each other to tell a story of customer satisfaction.

### **Key Takeaways**

* **The "Word Vector Mush" (Simple Averaging):** In this method, We retrieve the 300-dimensional embedding for every word in a sentence and simply **average** or **sum** them into a single 300-dimensional "sentence vector". This single vector is passed to a Softmax layer to predict a star rating ($1$–$5$).

    This method is **order-independent**. A review like *"Completely lacking in good taste"* contains the word "good". An averaging model may see the "good" vector and incorrectly predict a positive rating because it cannot process the negative context provided by the word "lacking."

* **The Sophisticated Approach (RNNs):** Instead of a simple sum, we feed the word embeddings into a **RNN** sequentially. The RNN processes the sequence one token at a time, using the hidden state from the final time step to make the star-rating prediction.

    Because RNN processes words in order, it recognizes that *"not good"* or *"lacking in good service"* carries a negative meaning, regardless of how many times the word "good" appears later in the string.

<div align='center'>
    <img src='assets/rnn_for_sentiment_classification.png' width=750px>
</div>


* **Generalization to "Unseen" Words:** If our labeled data contains the word "lacking" but a new customer uses the word "absent", a standard model without embeddings would likely fail. Since "lacking" and "absent" have nearly identical embeddings (learned from the massive 100-billion-word corpus), the RNN automatically knows how to handle the new word, even if it never appeared in our specific sentiment training set.

---

## **Debiasing Word Embeddings**

This section addresses the critical issue of **algorithmic bias** in word embeddings and outlines a systematic approach to mitigating it. As AI increasingly influences high-stakes decisions—such as loan approvals, job hiring, and criminal sentencing—ensuring these models do not perpetuate harmful societal stereotypes is essential.

### **High-Level Summary**

Word embeddings do more than just learn language; they absorb gender, ethnic, and socioeconomic biases found in human-generated text. These biases can lead to unfair treatment in real-world applications. For instance, a biased model might output the analogy *Man:Computer_Programmer* as *Woman:Homemaker*, reflecting historical inequities rather than objective definitions.

The core of the debiasing approach is the distinction between **definitional words** and **neutral words**. Definitional Words are terms like *grandfather* or *actress* where gender is a legitimate part of the definition. Neutral Words on the othe hand are professional or descriptive terms like *doctor*, *programmer*, or *babysitter* that should remain unbiased.

Researchers use a linear classifier to separate these categories, stripping bias from "Neutral" terms while preserving the identity of "Definitional" ones. While "neutralizing" a vector is a powerful mathematical tool, it remains one part of a broader, ongoing effort to build ethical AI systems.

<div align='center'>
    <img src='assets/debiasing_word_embedding.png' width=500px>
</div>

### **Key Takeaways**

Following the research by [Bolukbasi et al. (2016)](https://arxiv.org/abs/1607.06520), we can mitigate undesirable biases using this three-step workflow:

* **Step 1: Identify the Bias Direction:**
    * The goal is to isolate the specific "axis" in the embedding space that represents a bias (e.g., gender).
    * The algorithm takes several gendered pairs—such as $(e_{he} - e_{she})$ and $(e_{male} - e_{female})$—and calculates their differences.
    * Using Singular Value Decomposition (SVD), the model identifies a "Bias Direction" (a 1D subspace) and a "Non-Bias Subspace" (the remaining 299 dimensions in a 300D space).

    The figure below should help us visualize what neutralizing does. If we're using a 50D word embedding, the 50D space can be split into two parts: The bias-direction $g=e_{woman} - e_{man}$, and the remaining 49 dimensions, which is called $g_{\perp}$ here. In linear algebra, we say that the 49-dimensional $g_{\perp}$ is perpendicular (or "orthogonal") to $g$. The neutralization step takes a vector such as $e_{receptionist}$ and zeros out the component in the direction of $g$, giving us $e_{receptionist}^{debiased}$. 

    Even though $g_{\perp}$ is 49-dimensional, given the limitations of what we can draw on a 2D screen, it's illustrated using a 1-dimensional axis below.

    <div align='center'>
    <img src='assets/neutralizing.png' width=850px>
    </div>

    Given an input embedding $e$, we can use the following formulas to compute $e^{debiased}$: 

    $$e^{bias\_component} = \frac{e \cdot g}{||g||_2^2} g$$
    $$e^{debiased} = e - e^{bias\_component}$$

    We call $e^{bias\_component}$ as the projection of $e$ onto the direction $g$. 

    >**Note:** These debiasing algorithms are very helpful for reducing bias, but aren't perfect and don't eliminate all traces of bias. For example, one weakness of this implementation was that the bias direction $g$ was defined using only the pair of words _woman_ and _man_. As discussed earlier, if $g$ were defined by computing $g_1 = e_{woman} - e_{man}$; $g_2 = e_{mother} - e_{father}$; $g_3 = e_{girl} - e_{boy}$; and so on and averaging over them, we would obtain a better estimate of the "gender" dimension $D$ dimensional word embedding space.

* **Step 2: Neutralization:**
    * The goal is to ensure that words that *should* be neutral do not lean toward any specific bias.
    * For words like *doctor* or *receptionist*, the algorithm projects them onto the "Non-bias subspace".
    * This removes the mathematical component of the word that points toward a biased direction, effectively making the word equidistant from terms like "man" and "woman."

* **Step 3: Equalization:**
    * The goal is to maintain the meaning of "Definitional" words while ensuring they don't reinforce "unhealthy" distances to neutral words.
    * Even after neutralizing *babysitter*, it might still be mathematically closer to *grandmother* than *grandfather* due to other latent features in the training data.
    * The algorithm adjusts these pairs so they are exactly the same distance from the neutral axis. This ensures that the only difference between *grandmother* and *grandfather* is their gender, not their perceived "likelihood" to be a babysitter.

    The key idea behind equalization is to make sure that a particular pair of words are equidistant from the 49-dimensional $g_\perp$. The equalization step also ensures that the two equalized steps are now the same distance from $e_{receptionist}^{debiased}$, or from any other work that has been neutralized. Visually, this is how equalization works: 

    <div align='center'>
    <img src="assets/equalizing.png" width=850px>
    </div>

    The derivation of the linear algebra to do this is a bit more complex. (See Bolukbasi et al., 2016 in the References for details.) Here are the key equations:

    $$\mu = \frac{e_{w1} + e_{w2}}{2}$$
    
    $$\mu_{B} = \frac {\mu \cdot \text{bias\_axis}}{||\text{bias\_axis}||_2^2} * \text{bias\_axis}$$
    
    $$\mu_{\perp} = \mu - \mu_{B}$$
    
    $$e_{w1B} = \frac {e_{w1} \cdot \text{bias\_axis}}{||\text{bias\_axis}||_2^2} * \text{bias\_axis}$$
    
    $$e_{w2B} = \frac {e_{w2} \cdot \text{bias\_axis}}{||\text{bias\_axis}||_2^2} * \text{bias\_axis}$$
    
    $$e_{w1B}^{corrected} = \sqrt{{1 - ||\mu_{\perp} ||^2_2}} * \frac{e_{\text{w1B}} - \mu_B} {||e_{w1B} - \mu_B||_2}$$
    
    $$e_{w2B}^{corrected} = \sqrt{{1 - ||\mu_{\perp} ||^2_2}} * \frac{e_{\text{w2B}} - \mu_B} {||e_{w2B} - \mu_B||_2}$$
    $$e_1 = e_{w1B}^{corrected} + \mu_{\perp}$$ $$e_2 = e_{w2B}^{corrected} + \mu_{\perp}$$